In [6]:
from typing import List, Tuple, Dict, Set
from collections import defaultdict
import random
import pandas as pd
import numpy as np
import math

# --------------------
# Parameters & Sample Docs
# --------------------
k = 3  # shingle size (characters)
n_hashes = 20  # number of MinHash functions (controls accuracy vs efficiency)
random_seed = 42

# Example documents
docs: List[str] = [
    "Data mining discovers patterns in large datasets, uncovering associations and trends that may not be visible with traditional analysis techniques.",
    "Mining data helps find patterns in very large data sets, supporting decision-making processes in business, healthcare, and scientific research.",
    "Locality Sensitive Hashing finds similar documents efficiently by reducing high-dimensional vectors into compact signatures, enabling fast approximate nearest neighbor searches.",
    "MinHash approximates Jaccard similarity of sets, making it possible to compare massive collections of documents without constructing full pairwise similarity matrices.",
    "Efficient algorithms are crucial when working with massive datasets, as they allow practitioners to handle terabytes of text and multimedia data within reasonable time and memory constraints.",
    "Approximate methods trade accuracy for speed and scalability, which is essential when exact solutions are computationally infeasible in domains such as web search or large-scale recommendation.",
    "Hashing techniques are widely used in information retrieval systems, supporting fast indexing, deduplication, and similarity search across billions of documents.",
]

random.seed(random_seed)


## Shingling Functions

In [7]:
# --------------------
# Shingling Functions
# --------------------
def normalize_text(s: str, remove_spaces: bool = True) -> str:
    """
    Normalize a given text string by converting it to lowercase and optionally removing spaces.
    Args:
        s (str): The input string to normalize.
        remove_spaces (bool, optional): If True, all spaces will be removed from the string. Defaults to True.
    Returns:
        str: The normalized string.
    """

    s = s.lower()
    return s.replace(" ", "") if remove_spaces else s


def shingles_from_text(text: str, k: int) -> Set[str]:
    """
    Generates a set of k-length shingles (substrings) from the input text after normalization.

    Args:
        text (str): The input text to generate shingles from.
        k (int): The length of each shingle.

    Returns:
        Set[str]: A set containing all unique k-length shingles from the normalized text.

    Note:
        The text is normalized and spaces are removed before generating shingles.
    """
    t = normalize_text(text, remove_spaces=True)

    # Tao tap hop tat ca cac chuoi con lien tiep do dai k tu van ban da chuan hoa
    shingle_set = {t[i : i + k] for i in range(len(t) - k + 1)}
    
    return shingle_set


# Build shingle sets for all docs
doc_shingle_sets: List[Set[str]] = [shingles_from_text(d, k) for d in docs]

# Build shingle universe (i.e., the set of all shingles in all documents) and mapping from shingle to unique integer ID
shingle_universe: Set[str] = set().union(*doc_shingle_sets)
shingle_list: List[str] = sorted(shingle_universe)
shingle_to_id: Dict[str, int] = {s: i for i, s in enumerate(shingle_list)}


# Create shingle-docs incidence as list of sets (one set per shingle, containing doc indices)
# This is a sparse representation of the characteristic matrix
# (rows = shingles, columns = documents)
# Example: rows_as_sets[shingle_id] = {doc_id1, doc_id2, ...} indicates which documents contain that shingle
n_shingles = len(shingle_list)
n_docs = len(docs)
rows_as_sets = [set() for _ in range(n_shingles)]
for d_idx, sset in enumerate(doc_shingle_sets):
    for s in sset:
        rows_as_sets[shingle_to_id[s]].add(d_idx)


# Dense preview: show a human-readable version of the characteristic matrix as a dense DataFrame.
# This helps visualize which documents contain which shingles, making it easier to debug and understand the shingle-document relationships.
char_matrix_df = pd.DataFrame(
    0,
    index=[f"s_{i}" for i in range(n_shingles)],
    columns=[f"d_{j}" for j in range(n_docs)],
)
for r, doc_ids in enumerate(rows_as_sets):
    for d in doc_ids:
        char_matrix_df.iat[r, d] = 1

### MinHash Functions

In [8]:
# --------------------
# MinHash Functions (explicit callable hash functions)
# --------------------

# Assume a sufficiently large prime number is given
p = 10007


def make_hash_functions(n_hashes: int, p: int, n_shingles: int, rng: random.Random):
    """
    Generate a list of explicit hash functions and their (a, b) parameters for MinHash.

    Each hash function h(x) maps a shingle index x to an integer in [0, n_shingles-1] using:
        h(x) = ((a * x + b) % p) % n_shingles

    Args:
        n_hashes (int): Number of hash functions to generate.
        p (int): A large prime number for hashing.
        n_shingles (int): Number of unique shingles.
        rng (random.Random): Random number generator for reproducibility.

    Returns:
        funcs (List[Callable]): List of hash functions.
        params (List[Tuple[int, int]]): List of (a, b) parameter pairs for each hash function.
    """
    funcs = []
    params = []

    # Universal Hashing theo Slide 33 bai giang 03-lsh.pdf:
    # h_{a,b}(x) = ((a * x + b) mod p) mod N
    # voi a, b la cac so nguyen ngau nhien (a != 0 mod p, a in [1, p-1], b in [0, p-1]), p la so nguyen to > N.
    def create_hash_func(a: int, b: int):
        # Closure giup tranh loi late-binding trong Python
        return lambda x: ((a * x + b) % p) % n_shingles

    for _ in range(n_hashes):
        a = rng.randint(1, p - 1)
        b = rng.randint(0, p - 1)
        params.append((a, b))
        funcs.append(create_hash_func(a, b))
    
    return funcs, params


rng = random.Random(random_seed)
hash_functions, hash_params = make_hash_functions(n_hashes, p, n_shingles, rng)

### Signature Matrix 

In [9]:
# --------------------
# Signature Matrix
# --------------------
def compute_signature_matrix(
    rows_as_sets,
    n_shingles: int,
    n_docs: int,
    hash_functions: List,
) -> np.ndarray:
    """
    Computes the MinHash signature matrix for a set of documents.
    Args:
        rows_as_sets (List[Set[int]]): A list where each element is a set containing the indices of documents
            that contain the corresponding shingle.
        n_shingles (int): The total number of unique shingles.
        n_docs (int): The total number of documents.
        hash_functions (List[Callable[[int], int]]): A list of hash functions to be used for MinHashing.
    Returns:
        np.ndarray: A 2D NumPy array of shape (len(hash_functions), n_docs) representing the signature matrix,
            where each row corresponds to a hash function and each column corresponds to a document.
    """

    sig = np.full((len(hash_functions), n_docs), fill_value=math.inf, dtype=float)

    n_hashes = len(hash_functions)

    # 1. Ma tran duy tri gia tri cua cac hash functions cho tung shingle (kich thuoc: n_hashes x n_shingles)
    hash_matrix = np.empty((n_hashes, n_shingles), dtype=int)
    for i, h in enumerate(hash_functions):
        for r in range(n_shingles):
            hash_matrix[i, r] = h(r)

    # 2. Thuat toan One-Pass cap nhat Signature Matrix:
    # - Khoi tao tat ca M(i, c) = infinity (da khoi tao o tren)
    # - Quet tung hang r tim cac cot c co gia tri 1 (docs_with_shingle = rows_as_sets[r])
    # - Voi moi ham bam h_i: neu h_i(r) < M(i, c) thi M(i, c) <- h_i(r)
    for r in range(n_shingles):
        docs_with_shingle = rows_as_sets[r]
        if not docs_with_shingle:
            continue
        h_vals = hash_matrix[:, r]
        for c in docs_with_shingle:
            for i in range(n_hashes):
                if h_vals[i] < sig[i, c]:
                    sig[i, c] = h_vals[i]
    
    return sig


signature = compute_signature_matrix(rows_as_sets, n_shingles, n_docs, hash_functions)
sig_df = pd.DataFrame(signature, columns=[f"d_{j}" for j in range(n_docs)])

### Jaccard vs MinHash 

In [10]:

# --------------------
# Jaccard vs MinHash
# --------------------
def jaccard(a: Set[str], b: Set[str]) -> float:
    if not a and not b:
        return 1.0
    return len(a & b) / max(1, len(a | b))


def minhash_sim_from_signatures(sig: np.ndarray, i: int, j: int) -> float:
    return float((sig[:, i] == sig[:, j]).sum()) / sig.shape[0]


pairs = []
for i in range(n_docs):
    for j in range(i + 1, n_docs):
        true_j = jaccard(doc_shingle_sets[i], doc_shingle_sets[j])
        est_j = minhash_sim_from_signatures(signature, i, j)
        pairs.append((f"d_{i}", f"d_{j}", true_j, est_j))

sim_df = pd.DataFrame(pairs, columns=["doc_i", "doc_j", "true_jaccard", "minhash_est"])

# Preview outputs
print("Characteristic matrix preview:")
print(char_matrix_df)
print("\nSignature matrix preview:")
print(sig_df)
print("\nJaccard vs MinHash estimate:")
print(sim_df)


Characteristic matrix preview:
       d_0  d_1  d_2  d_3  d_4  d_5  d_6
s_0      0    1    0    0    0    0    1
s_1      0    0    0    0    1    0    0
s_2      0    0    0    0    0    0    1
s_3      0    0    1    0    0    0    0
s_4      0    1    0    0    0    0    0
...    ...  ...  ...  ...  ...  ...  ...
s_625    0    0    1    0    0    0    1
s_626    1    0    0    0    0    0    0
s_627    0    0    0    0    0    0    1
s_628    0    0    0    0    1    0    0
s_629    0    0    0    0    0    0    1

[630 rows x 7 columns]

Signature matrix preview:
     d_0   d_1  d_2   d_3   d_4   d_5   d_6
0    8.0   0.0  3.0   4.0   4.0   0.0   3.0
1    0.0   5.0  3.0   1.0   1.0   2.0   1.0
2   10.0  10.0  5.0   5.0   5.0  10.0  10.0
3    5.0  11.0  5.0   1.0   1.0   1.0   4.0
4    0.0   2.0  2.0   0.0   0.0   2.0   2.0
5   13.0   2.0  2.0   4.0   0.0   1.0   2.0
6    1.0   2.0  3.0   2.0   7.0   3.0  14.0
7    1.0   1.0  4.0   1.0   1.0   9.0   4.0
8    0.0   0.0  0.0   6.0   3.

In [11]:
# --------------------
# Benchmark: Memory & Computation Benefits of MinHash Signatures
# --------------------
import time
import string


def bytes_human(num_bytes: int) -> str:
    units = ["B", "KB", "MB", "GB", "TB"]
    i = 0
    x = float(num_bytes)
    while x >= 1024 and i < len(units) - 1:
        x /= 1024.0
        i += 1
    return f"{x:.2f} {units[i]}"


def characteristic_bool_matrix(
    rows_as_sets: List[set], n_shingles: int, n_docs: int
) -> np.ndarray:
    mat = np.zeros((n_shingles, n_docs), dtype=bool)
    for r, docs_with in enumerate(rows_as_sets):
        for d in docs_with:
            mat[r, d] = True
    return mat


# Memory comparison on the CURRENT corpus
char_bool = characteristic_bool_matrix(rows_as_sets, n_shingles, n_docs)
mem_char_theoretical_bits = (n_shingles * n_docs) / 8  # bit-packed ideal
mem_char_bool = char_bool.nbytes  # practical bool matrix
mem_sig_int32 = signature.astype(np.int32).nbytes  # signature matrix as int32

print("=== Memory Footprint (current corpus) ===")
print(
    f"Characteristic matrix (ideal bit-packed): {bytes_human(int(mem_char_theoretical_bits))}"
)
print(f"Characteristic matrix (numpy bool):        {bytes_human(mem_char_bool)}")
print(
    f"Signature matrix (int32, {n_hashes}×{n_docs}):     {bytes_human(mem_sig_int32)}\n"
)

# Computation: all-pairs similarity cost (current corpus)
start = time.time()
for i in range(n_docs):
    for j in range(i + 1, n_docs):
        _ = jaccard(doc_shingle_sets[i], doc_shingle_sets[j])
exact_time = time.time() - start

start = time.time()
for i in range(n_docs):
    for j in range(i + 1, n_docs):
        _ = (signature[:, i] == signature[:, j]).mean()
minhash_time = time.time() - start

print("=== Computation Time (current corpus) ===")
print(f"All-pairs Jaccard on sets: {exact_time:.6f} s")
print(f"All-pairs MinHash estimate: {minhash_time:.6f} s")

=== Memory Footprint (current corpus) ===
Characteristic matrix (ideal bit-packed): 551.00 B
Characteristic matrix (numpy bool):        4.31 KB
Signature matrix (int32, 20×7):     560.00 B

=== Computation Time (current corpus) ===
All-pairs Jaccard on sets: 0.000374 s
All-pairs MinHash estimate: 0.000371 s
